# 03 Gold (A): analysis + tables

**Owner:** Person A
**Builds:** `gold.dim_customers`, `gold.fact_orders`, `gold.fact_order_items`, `gold.mart_customer_rfm`

**Workflow per question:** question -> explore on silver (read-only) -> finding and decision -> build the gold table -> checks.
The analysis decides what each gold table must contain.

**Rules**
- Exploration queries are read-only; ad-hoc joins on silver are fine there.
- Saved tables follow the gold rule: aggregate one-to-many tables to one row per key BEFORE joining.
- Every build cell is safe to re-run (`DROP TABLE IF EXISTS` first).
- Each finding is 2-3 sentences with numbers.

| Block | Question | Gold table |
|---|---|---|
| A1 | Who are the customers? | `dim_customers` |
| A2 | How does delivery perform? | `fact_orders` |
| A3 | How do customers pay? | payment columns in `fact_orders` |
| A4 | How do revenue and orders change over time? | `fact_order_items` |
| A5 | Who are the best and worst customers? (RFM) | `mart_customer_rfm` |

## A0. Setup

In [4]:
# Importing libraries
import pandas as pd
from sqlalchemy import text
import sys
from pathlib import Path

# db.py lives in notebooks/, two levels above this notebook's folder
# (adjust .parent chain if this notebook is placed in a different folder)
NOTEBOOKS_DIR = Path.cwd().parent.parent
sys.path.append(str(NOTEBOOKS_DIR))

from db import get_engine
engine = get_engine()


In [5]:
def q(sql):
    """Run a SELECT and return the result as a DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

def run(sql):
    """Run CREATE/DROP/INSERT and commit."""
    with engine.begin() as conn:
        conn.execute(text(sql))

---

## A1. Who are the customers?
- **Explore:** customers vs unique people, repeat-buyer share, customers per state.
- **Decision:** what `dim_customers` must contain.
- **Build:** `gold.dim_customers`, then checks.

**`gold.dim_customers` spec**
- **Grain:** one row per `customer_id`.
- **Base:** `silver.customers`; **Joins:** none.
- **Columns:** `customer_id`, `customer_unique_id`, `customer_zip_code_prefix`, `customer_city`, `customer_state`.
- **Note:** a real person is `customer_unique_id` (one person can have several `customer_id` values). Use `DISTINCTCOUNT(customer_unique_id)` in Power BI for customer counts.

In [6]:
# Customer ids vs real people, and repeat-buyer share
q("""
WITH per_person AS (
    SELECT c.customer_unique_id, COUNT(DISTINCT o.order_id) AS orders
    FROM silver.customers c
    JOIN silver.orders o ON o.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
)
SELECT COUNT(*)                                          AS unique_people,
       SUM(CASE WHEN orders >= 2 THEN 1 ELSE 0 END)      AS repeat_buyers,
       100.0 * SUM(CASE WHEN orders >= 2 THEN 1 ELSE 0 END) / COUNT(*) AS repeat_pct
FROM per_person
""")

,unique_people,repeat_buyers,repeat_pct
0,96096,2997,3.118756


In [7]:
# Customers per state (top 10)
q("""
SELECT TOP 10 customer_state, COUNT(DISTINCT customer_unique_id) AS people
FROM silver.customers
GROUP BY customer_state
ORDER BY people DESC
""")

,customer_state,people
0,SP,40302
1,RJ,12384
2,MG,11259
3,RS,5277
4,PR,4882
5,SC,3534
6,BA,3277
7,DF,2075
8,ES,1964
9,GO,1952


**Finding:** The data has 96,096 unique people, and only 2,997 of them (3.1%) ordered more than once, so almost all customers are one-time buyers. Customers are concentrated in a few states: São Paulo has 40,302 people (about 42%), followed by RJ (12,384) and MG (11,259), and these three states together hold about 67% of all customers.

**Decision:** keep one row per `customer_id` and keep `customer_unique_id` as a column for customer counts and RFM. Because repeat buyers are so rare, the RFM frequency score will use a simple rule (1, 2, 3+ orders) instead of 5 equal groups, and state-level visuals should be read with SP's dominance in mind.

In [8]:
# Rebuild gold.dim_customers (safe to re-run)
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.dim_customers"))
    conn.execute(text("""
        SELECT
            customer_id,
            customer_unique_id,
            customer_zip_code_prefix,
            customer_city,
            customer_state
        INTO gold.dim_customers
        FROM silver.customers
    """))

#### Checks: dim_customers

In [9]:
# 1. Row count must equal silver
q("""
SELECT (SELECT COUNT(*) FROM silver.customers)    AS silver_rows,
       (SELECT COUNT(*) FROM gold.dim_customers)  AS gold_rows
""")

,silver_rows,gold_rows
0,99441,99441


In [10]:
# 2. customer_id must be unique and not null (total_rows = distinct_ids, null_ids = 0)
q("""
SELECT COUNT(*)                     AS total_rows,
       COUNT(DISTINCT customer_id)  AS distinct_ids,
       SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS null_ids
FROM gold.dim_customers
""")

,total_rows,distinct_ids,null_ids
0,99441,99441,0


In [11]:
# 3. Real customers vs customer_id values (informational: fewer unique people expected)
q("""
SELECT COUNT(DISTINCT customer_unique_id) AS unique_customers,
       COUNT(*)                           AS customer_ids
FROM gold.dim_customers
""")

,unique_customers,customer_ids
0,96096,99441


---

## A2. How does delivery perform?
- **Explore:** share of delivered orders, delivery days, late share by state.
- **Decision:** which delivery columns `fact_orders` must hold.
- **Build:** `gold.fact_orders` (in small steps with staging tables), then checks.

**`gold.fact_orders` spec**
- **Grain:** one row per order. **Base:** `silver.orders`.
- **Columns:** `order_id`, `customer_id`, `order_status`, `order_date` (DATE), `approved_date`, `delivered_carrier_date`, `delivered_customer_date`, `estimated_delivery_date`, `delivery_days`, `delay_days`, `is_late` (BIT), `total_payment`, `payment_installments_max`, `main_payment_type`.
- **Logic:** `delivery_days` = delivered date minus purchase date; `delay_days` = delivered date minus estimated date; `is_late` = `delay_days > 0`. Delivery columns stay `NULL` for undelivered orders (`is_late` is NULL, not 0).

In [12]:
# Order status mix and how many orders have a delivered date
q("""
SELECT order_status,
       COUNT(*) AS orders,
       SUM(CASE WHEN order_delivered_customer_date IS NOT NULL THEN 1 ELSE 0 END) AS with_delivered_date
FROM silver.orders
GROUP BY order_status
ORDER BY orders DESC
""")

,order_status,orders,with_delivered_date
0,delivered,96478,96470
1,shipped,1107,0
2,canceled,625,6
3,unavailable,609,0
4,invoiced,314,0
5,processing,301,0
6,created,5,0
7,approved,2,0


In [13]:
# Late share and delivery days by customer state (delivered orders only, ad hoc join on silver)
q("""
SELECT c.customer_state,
       COUNT(*)                                        AS delivered_orders,
       AVG(CAST(DATEDIFF(day, o.order_purchase_timestamp,
                              o.order_delivered_customer_date) AS FLOAT)) AS avg_delivery_days,
       100.0 * AVG(CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
                        THEN 1.0 ELSE 0.0 END)         AS late_pct
FROM silver.orders o
JOIN silver.customers c ON c.customer_id = o.customer_id
WHERE o.order_delivered_customer_date IS NOT NULL
GROUP BY c.customer_state
ORDER BY late_pct DESC
""")

,customer_state,delivered_orders,avg_delivery_days,late_pct
0,AL,397,24.501259,23.9294
1,MA,717,21.511855,19.6652
2,PI,476,19.394958,15.9663
3,CE,1279,21.200156,15.3244
4,SE,335,21.462687,15.2238
5,BA,3256,19.278563,14.0356
6,RJ,12353,15.237675,13.4704
7,TO,274,17.598540,12.7737
8,PA,946,23.725159,12.3678
9,ES,1995,15.723810,12.2305


**Finding:** Of 99,441 orders, 96,476 (97%) have a delivered date, and 2,965 (3.0%) have none, mostly shipped, canceled, unavailable, invoiced and processing orders. Across delivered orders, about 8.1% arrive later than estimated. Delivery is clearly worse in the north and northeast: AL has the highest late rate (23.9%, 24.5 days on average), followed by MA (19.7%), PI (16.0%) and CE (15.3%), while SP (5.9%, 8.7 days) and MG (5.6%) are among the best. The slowest in raw days are small northern states such as RR (29.3 days) and AP (27.2 days), but they have very few orders (41 and 67), so those averages are unreliable.

**Decision:** keep `delivery_days`, `delay_days` and `is_late` as NULL for undelivered orders, so averages and late % only count delivered orders. Six orders are canceled but have a delivered date; keep them as they are and let Power BI filter by `order_status`. State-level visuals should show the order count next to late %, and small states (under about 100 delivered orders) should be flagged or filtered.

In [14]:
# Check column names before building
q("""
SELECT TABLE_NAME, COLUMN_NAME, DATA_TYPE
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'silver' AND TABLE_NAME IN ('orders', 'order_payments')
ORDER BY TABLE_NAME, ORDINAL_POSITION
""")

,TABLE_NAME,COLUMN_NAME,DATA_TYPE
0,order_payments,order_id,varchar
1,order_payments,payment_sequential,smallint
2,order_payments,payment_type,varchar
3,order_payments,payment_installments,smallint
4,order_payments,payment_value,decimal
5,orders,order_id,varchar
6,orders,customer_id,varchar
7,orders,order_status,varchar
8,orders,order_purchase_timestamp,datetime2
9,orders,order_approved_at,datetime2


#### A3 first: payments feed `fact_orders`. Steps 1 and 2 below are the payment part (see A3 explore after the build).

In [15]:
# Step 1: one row per order with total payment and max installments
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold._tmp_pay"))
    conn.execute(text("""
        SELECT order_id,
               SUM(payment_value)        AS total_payment,
               MAX(payment_installments) AS payment_installments_max
        INTO gold._tmp_pay
        FROM silver.order_payments
        GROUP BY order_id
    """))

q("SELECT COUNT(*) AS rows_, COUNT(DISTINCT order_id) AS orders FROM gold._tmp_pay")

,rows_,orders
0,99440,99440


In [16]:
# Step 2: main payment type = type with the highest value in the order
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold._tmp_main_type"))
    conn.execute(text("""
        WITH t AS (
            SELECT order_id, payment_type,
                   ROW_NUMBER() OVER (
                       PARTITION BY order_id
                       ORDER BY SUM(payment_value) DESC, payment_type
                   ) AS rn
            FROM silver.order_payments
            GROUP BY order_id, payment_type
        )
        SELECT order_id, payment_type AS main_payment_type
        INTO gold._tmp_main_type
        FROM t
        WHERE rn = 1
    """))

q("SELECT COUNT(*) AS rows_, COUNT(DISTINCT order_id) AS orders FROM gold._tmp_main_type")

,rows_,orders
0,99440,99440


In [17]:
# Step 3: orders with delivery_days and delay_days (NULL if not delivered)
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold._tmp_base"))
    conn.execute(text("""
        SELECT order_id,
               customer_id,
               order_status,
               CAST(order_purchase_timestamp AS DATE) AS order_date,
               order_approved_at                      AS approved_date,
               order_delivered_carrier_date           AS delivered_carrier_date,
               order_delivered_customer_date          AS delivered_customer_date,
               order_estimated_delivery_date          AS estimated_delivery_date,
               CASE WHEN order_delivered_customer_date IS NOT NULL
                    THEN DATEDIFF(day, CAST(order_purchase_timestamp AS DATE),
                                       CAST(order_delivered_customer_date AS DATE)) END AS delivery_days,
               CASE WHEN order_delivered_customer_date IS NOT NULL
                         AND order_estimated_delivery_date IS NOT NULL
                    THEN DATEDIFF(day, CAST(order_estimated_delivery_date AS DATE),
                                       CAST(order_delivered_customer_date AS DATE)) END AS delay_days
        INTO gold._tmp_base
        FROM silver.orders
    """))

q("SELECT COUNT(*) AS rows_ FROM gold._tmp_base")

,rows_
0,99441


In [18]:
# Step 4: final table = base + payment total + main payment type
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.fact_orders"))
    conn.execute(text("""
        SELECT b.order_id,
               b.customer_id,
               b.order_status,
               b.order_date,
               b.approved_date,
               b.delivered_carrier_date,
               b.delivered_customer_date,
               b.estimated_delivery_date,
               b.delivery_days,
               b.delay_days,
               -- NULL (not 0) when delay_days is unknown
               CAST(CASE WHEN b.delay_days IS NULL THEN NULL
                         WHEN b.delay_days > 0 THEN 1
                         ELSE 0 END AS BIT)  AS is_late,
               p.total_payment,
               p.payment_installments_max,
               m.main_payment_type
        INTO gold.fact_orders
        FROM gold._tmp_base b
        LEFT JOIN gold._tmp_pay p       ON p.order_id = b.order_id
        LEFT JOIN gold._tmp_main_type m ON m.order_id = b.order_id
    """))

q("""
SELECT (SELECT COUNT(*) FROM silver.orders)    AS silver_rows,
       (SELECT COUNT(*) FROM gold.fact_orders) AS gold_rows
""")

,silver_rows,gold_rows
0,99441,99441


In [19]:
# Step 5: drop the staging tables
with engine.begin() as conn:
    for t in ["_tmp_pay", "_tmp_main_type", "_tmp_base"]:
        conn.execute(text(f"DROP TABLE IF EXISTS gold.{t}"))

#### Checks: fact_orders

In [20]:
# 1. order_id must be unique (total_rows = distinct_ids)
q("""
SELECT COUNT(*) AS total_rows, COUNT(DISTINCT order_id) AS distinct_ids
FROM gold.fact_orders
""")

,total_rows,distinct_ids
0,99441,99441


In [21]:
# 2. Total payment vs silver (explain small differences in docs/gold_log.md)
q("""
SELECT (SELECT SUM(payment_value) FROM silver.order_payments) AS silver_total,
       (SELECT SUM(total_payment) FROM gold.fact_orders)      AS gold_total
""")

,silver_total,gold_total
0,16008872.12,16008872.12


In [22]:
# 3. Delivery columns must be NULL for undelivered orders, never 0
q("""
SELECT order_status,
       COUNT(*)                                                         AS orders,
       SUM(CASE WHEN delivered_customer_date IS NULL THEN 1 ELSE 0 END) AS undelivered,
       SUM(CASE WHEN is_late IS NULL THEN 1 ELSE 0 END)                 AS is_late_null,
       SUM(CASE WHEN is_late = 1 THEN 1 ELSE 0 END)                     AS late_orders
FROM gold.fact_orders
GROUP BY order_status
ORDER BY orders DESC
""")

,order_status,orders,undelivered,is_late_null,late_orders
0,delivered,96478,8,8,6534
1,shipped,1107,1107,1107,0
2,canceled,625,619,619,1
3,unavailable,609,609,609,0
4,invoiced,314,314,314,0
5,processing,301,301,301,0
6,created,5,5,5,0
7,approved,2,2,2,0


---

## A3. How do customers pay?
- **Explore:** share of orders and value by payment type, installments by type.
- **Decision:** payment columns in `fact_orders` (built above) are enough; no separate payments table in gold.

In [23]:
# Payment type mix from the final gold table
q("""
SELECT main_payment_type,
       COUNT(*)                                           AS orders,
       100.0 * COUNT(*) / SUM(COUNT(*)) OVER ()           AS orders_pct,
       AVG(total_payment)                                 AS avg_order_value,
       AVG(CAST(payment_installments_max AS FLOAT))       AS avg_installments
FROM gold.fact_orders
GROUP BY main_payment_type
ORDER BY orders DESC
""")

,main_payment_type,orders,orders_pct,avg_order_value,avg_installments
0,credit_card,74935,75.356241,167.283248,3.555428
1,boleto,19784,19.895214,145.034435,1.000000
2,voucher,3191,3.208938,121.028147,1.150423
3,debit_card,1527,1.535584,142.724158,1.000000
4,not_defined,3,0.003017,0.000000,1.000000
5,None,1,0.001006,NaN,NaN


**Finding:** Credit card is the main payment type for 74,935 orders (75.4%), followed by boleto with 19,784 (19.9%), voucher with 3,191 (3.2%) and debit card with 1,527 (1.5%). Credit card orders also have the highest average payment (167.28) and use 3.6 installments on average, while boleto and debit card are always paid in a single installment and have lower averages (145.03 and 142.72). Four orders have no usable payment info: three are `not_defined` with a payment value of 0, and one has no payment record at all.

**Decision:** payment columns in `fact_orders` (`total_payment`, `payment_installments_max`, `main_payment_type`) are enough, so there is no separate payments table in gold. Keep the four odd orders in the table and let Power BI filter them by `main_payment_type`, since removing them in SQL would break the row count against `silver.orders`.

---

## A4. How do revenue and orders change over time?
- **Explore:** monthly revenue and orders on silver, revenue by order status, orphan check.
- **Decision:** what `fact_order_items` must contain.
- **Build:** `gold.fact_order_items`, then checks.

**`gold.fact_order_items` spec**
- **Grain:** one row per order item (`order_id` + `order_item_id`). **Base:** `silver.order_items`.
- **Joins:** `LEFT JOIN silver.orders` on `order_id` (one row per order, so no duplication) for `customer_id`, `order_date`, `order_status`.
- **Do NOT join** products, sellers or payments. Keep `product_id` and `seller_id` as keys.
- **Columns:** `order_id`, `order_item_id`, `product_id`, `seller_id`, `customer_id`, `order_date`, `order_status`, `price`, `freight_value`, `item_total` (price + freight).
- **Note:** keep payments out of this table; order-level amounts live in `fact_orders`, otherwise they repeat on every item and inflate totals.

In [24]:
# Monthly revenue and orders, joined ad hoc on silver (read-only)
q("""
SELECT FORMAT(o.order_purchase_timestamp, 'yyyy-MM') AS month,
       COUNT(DISTINCT i.order_id)                    AS orders,
       COUNT(*)                                      AS items,
       SUM(i.price)                                  AS revenue
FROM silver.order_items i
JOIN silver.orders o ON o.order_id = i.order_id
GROUP BY FORMAT(o.order_purchase_timestamp, 'yyyy-MM')
ORDER BY month
""")

,month,orders,items,revenue
0,2016-09,3,6,267.36
1,2016-10,308,363,49507.66
2,2016-12,1,1,10.90
3,2017-01,789,955,120312.87
4,2017-02,1733,1951,247303.02
5,2017-03,2641,3000,374344.30
6,2017-04,2391,2684,359927.23
7,2017-05,3660,4136,506071.14
8,2017-06,3217,3583,433038.60
9,2017-07,3969,4519,498031.48


In [25]:
# Do canceled/unavailable orders carry item revenue?
q("""
SELECT o.order_status,
       COUNT(DISTINCT i.order_id) AS orders,
       SUM(i.price)               AS revenue
FROM silver.order_items i
JOIN silver.orders o ON o.order_id = i.order_id
GROUP BY o.order_status
ORDER BY revenue DESC
""")

,order_status,orders,revenue
0,delivered,96478,13221498.11
1,shipped,1106,150727.44
2,canceled,461,95235.27
3,invoiced,312,61526.37
4,processing,301,60439.22
5,unavailable,6,2007.69
6,approved,2,209.60


In [26]:
# Items whose order is missing in silver.orders (must be 0)
q("""
SELECT COUNT(*) AS orphan_items
FROM silver.order_items i
LEFT JOIN silver.orders o ON o.order_id = i.order_id
WHERE o.order_id IS NULL
""")

,orphan_items
0,0


**Finding:** _write 2-3 sentences with numbers: how revenue moves by month, how much revenue sits in canceled/unavailable orders, orphan count._

**Decision:** keep all item rows and keep `order_status` in the fact table, so Power BI can filter statuses with DAX instead of losing rows in SQL. Keep `price` and `freight_value` separate and add `item_total`.

In [27]:
# Build gold.fact_order_items (safe to re-run)
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.fact_order_items"))
    conn.execute(text("""
        SELECT i.order_id,
               i.order_item_id,
               i.product_id,
               i.seller_id,
               o.customer_id,
               CAST(o.order_purchase_timestamp AS DATE) AS order_date,
               o.order_status,
               i.price,
               i.freight_value,
               i.price + i.freight_value                AS item_total
        INTO gold.fact_order_items
        FROM silver.order_items i
        LEFT JOIN silver.orders o ON o.order_id = i.order_id
    """))

#### Checks: fact_order_items

In [28]:
# 1. Row count must equal silver
q("""
SELECT (SELECT COUNT(*) FROM silver.order_items)     AS silver_rows,
       (SELECT COUNT(*) FROM gold.fact_order_items)  AS gold_rows
""")

,silver_rows,gold_rows
0,112650,112650


In [29]:
# 2. Key (order_id + order_item_id) must be unique, and totals must match silver
q("""
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT CONCAT(order_id, '-', order_item_id)) AS distinct_keys,
       (SELECT SUM(price) FROM silver.order_items)          AS silver_price,
       SUM(price)                                           AS gold_price
FROM gold.fact_order_items
""")

,total_rows,distinct_keys,silver_price,gold_price
0,112650,112650,13591643.7,13591643.7


In [30]:
# 3. No NULL customer_id or order_date (would mean a missing order; both must be 0)
q("""
SELECT SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS null_customer,
       SUM(CASE WHEN order_date  IS NULL THEN 1 ELSE 0 END) AS null_date
FROM gold.fact_order_items
""")

,null_customer,null_date
0,0,0


---

## A5. Who are the best and worst customers? (RFM)
- **Grain:** one row per `customer_unique_id`; **delivered orders only**.
- **Joins:** `gold.fact_orders` + `gold.dim_customers` (to get `customer_unique_id`) + `gold.fact_order_items` aggregated to one row per `order_id` first.
- **Columns:** `customer_unique_id`, `recency_days`, `frequency`, `monetary`, `r_score` (1-5), `f_score` (1-3), `m_score` (1-5), `segment`.
- **Logic:** reference date = last delivered order + 1 day. R and M use `NTILE(5)`. F: 1 order = 1, 2 orders = 2, 3+ = 3.
- **Segments:** Champions (f >= 2 and r >= 4), Loyal (f >= 2), New high value (r >= 4 and m >= 4), New (r >= 4), Lost (r = 1), Needs attention (rest).

In [31]:
from IPython.display import display

# 1. Orders per person (delivered only)
display(q("""
WITH p AS (
    SELECT c.customer_unique_id, COUNT(*) AS orders
    FROM gold.fact_orders f
    JOIN gold.dim_customers c ON c.customer_id = f.customer_id
    WHERE f.order_status = 'delivered'
    GROUP BY c.customer_unique_id
)
SELECT CASE WHEN orders >= 3 THEN '3+' ELSE CAST(orders AS VARCHAR(2)) END AS orders_bucket,
       COUNT(*) AS people
FROM p
GROUP BY CASE WHEN orders >= 3 THEN '3+' ELSE CAST(orders AS VARCHAR(2)) END
ORDER BY orders_bucket
"""))

# 2. Spend per person and date range
display(q("""
WITH p AS (
    SELECT c.customer_unique_id, SUM(i.price) AS monetary
    FROM gold.fact_order_items i
    JOIN gold.fact_orders f   ON f.order_id = i.order_id
    JOIN gold.dim_customers c ON c.customer_id = f.customer_id
    WHERE f.order_status = 'delivered'
    GROUP BY c.customer_unique_id
)
SELECT DISTINCT
       PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY monetary) OVER () AS median_spend,
       PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY monetary) OVER () AS p90_spend,
       AVG(monetary) OVER ()                                          AS avg_spend,
       (SELECT MIN(order_date) FROM gold.fact_orders WHERE order_status = 'delivered') AS first_order,
       (SELECT MAX(order_date) FROM gold.fact_orders WHERE order_status = 'delivered') AS last_order
FROM p
"""))

,orders_bucket,people
0,1,90557
1,2,2573
2,3+,228


,median_spend,p90_spend,avg_spend,first_order,last_order
0,89.73,279.99,141.621479,2016-09-15,2018-08-29


In [ ]:
# Build gold.mart_customer_rfm in one query (safe to re-run)
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.mart_customer_rfm"))
    conn.execute(text("""
        WITH items AS (
            -- Aggregate items to one row per order BEFORE joining
            SELECT order_id, SUM(price) AS order_value
            FROM gold.fact_order_items
            GROUP BY order_id
        ),
        ord AS (
            SELECT c.customer_unique_id, f.order_id, f.order_date,
                   COALESCE(i.order_value, 0) AS order_value
            FROM gold.fact_orders f
            JOIN gold.dim_customers c ON c.customer_id = f.customer_id
            LEFT JOIN items i         ON i.order_id = f.order_id
            WHERE f.order_status = 'delivered'
        ),
        base AS (
            -- Reference date = day after the last order in the data
            SELECT customer_unique_id,
                   DATEDIFF(day, MAX(order_date), (SELECT DATEADD(day, 1, MAX(order_date)) FROM ord)) AS recency_days,
                   COUNT(DISTINCT order_id) AS frequency,
                   SUM(order_value)         AS monetary
            FROM ord
            GROUP BY customer_unique_id
        ),
        s AS (
            SELECT *,
                   NTILE(5) OVER (ORDER BY recency_days DESC, customer_unique_id) AS r_score,
                   CASE WHEN frequency >= 3 THEN 3 WHEN frequency = 2 THEN 2 ELSE 1 END AS f_score,
                   NTILE(5) OVER (ORDER BY monetary ASC, customer_unique_id)      AS m_score
            FROM base
        )
        SELECT customer_unique_id, recency_days, frequency, monetary,
               r_score, f_score, m_score,
               CASE WHEN f_score >= 2 AND r_score >= 4 THEN 'Champions'
                    WHEN f_score >= 2                  THEN 'Loyal'
                    WHEN r_score >= 4 AND m_score >= 4 THEN 'New high value'
                    WHEN r_score >= 4                  THEN 'New'
                    WHEN r_score = 1                   THEN 'Lost'
                    ELSE 'Needs attention' END         AS segment
        INTO gold.mart_customer_rfm
        FROM s
    """))

In [33]:
# 1. One row per person (all three numbers must match)
display(q("""
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT customer_unique_id) AS distinct_people,
       (SELECT COUNT(DISTINCT c.customer_unique_id)
        FROM gold.fact_orders f
        JOIN gold.dim_customers c ON c.customer_id = f.customer_id
        WHERE f.order_status = 'delivered') AS expected_people
FROM gold.mart_customer_rfm
"""))

# 2. Score ranges (r and m: 1-5, f: 1-3) and monetary total vs delivered item revenue
display(q("""
SELECT MIN(r_score) AS r_min, MAX(r_score) AS r_max,
       MIN(f_score) AS f_min, MAX(f_score) AS f_max,
       MIN(m_score) AS m_min, MAX(m_score) AS m_max,
       SUM(monetary) AS mart_total,
       (SELECT SUM(i.price) FROM gold.fact_order_items i
        JOIN gold.fact_orders f ON f.order_id = i.order_id
        WHERE f.order_status = 'delivered') AS items_total
FROM gold.mart_customer_rfm
"""))

# 3. Segment sizes (use for the finding)
display(q("""
SELECT segment,
       COUNT(*)                                 AS people,
       100.0 * COUNT(*) / SUM(COUNT(*)) OVER () AS people_pct,
       AVG(monetary)                            AS avg_spend,
       AVG(CAST(recency_days AS FLOAT))         AS avg_recency_days
FROM gold.mart_customer_rfm
GROUP BY segment
ORDER BY people DESC
"""))

,total_rows,distinct_people,expected_people
0,93358,93358,93358


,r_min,r_max,f_min,f_max,m_min,m_max,mart_total,items_total
0,1,5,1,3,1,5,13221498.11,13221498.11


,segment,people,people_pct,avg_spend,avg_recency_days
0,Needs attention,36180,38.754044,135.576197,268.502736
1,New,21662,23.203153,55.520907,90.122380
2,Lost,18236,19.533409,138.067672,473.989197
3,New high value,14479,15.509115,267.107298,92.209200
4,Loyal,1600,1.713833,252.031875,319.691875
5,Champions,1201,1.286446,270.739175,89.075770


In [34]:
# 1. One row per person (all three numbers must match)
display(q("""
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT customer_unique_id) AS distinct_people,
       (SELECT COUNT(DISTINCT c.customer_unique_id)
        FROM gold.fact_orders f
        JOIN gold.dim_customers c ON c.customer_id = f.customer_id
        WHERE f.order_status = 'delivered') AS expected_people
FROM gold.mart_customer_rfm
"""))

# 2. Score ranges (r and m: 1-5, f: 1-3) and monetary total vs delivered item revenue
display(q("""
SELECT MIN(r_score) AS r_min, MAX(r_score) AS r_max,
       MIN(f_score) AS f_min, MAX(f_score) AS f_max,
       MIN(m_score) AS m_min, MAX(m_score) AS m_max,
       SUM(monetary) AS mart_total,
       (SELECT SUM(i.price) FROM gold.fact_order_items i
        JOIN gold.fact_orders f ON f.order_id = i.order_id
        WHERE f.order_status = 'delivered') AS items_total
FROM gold.mart_customer_rfm
"""))

# 3. Segment sizes (use for the finding)
display(q("""
SELECT segment,
       COUNT(*)                                 AS people,
       100.0 * COUNT(*) / SUM(COUNT(*)) OVER () AS people_pct,
       AVG(monetary)                            AS avg_spend,
       AVG(CAST(recency_days AS FLOAT))         AS avg_recency_days
FROM gold.mart_customer_rfm
GROUP BY segment
ORDER BY people DESC
"""))

,total_rows,distinct_people,expected_people
0,93358,93358,93358


,r_min,r_max,f_min,f_max,m_min,m_max,mart_total,items_total
0,1,5,1,3,1,5,13221498.11,13221498.11


,segment,people,people_pct,avg_spend,avg_recency_days
0,Needs attention,36180,38.754044,135.576197,268.502736
1,New,21662,23.203153,55.520907,90.122380
2,Lost,18236,19.533409,138.067672,473.989197
3,New high value,14479,15.509115,267.107298,92.209200
4,Loyal,1600,1.713833,252.031875,319.691875
5,Champions,1201,1.286446,270.739175,89.075770


**Finding:** Only 2,801 of 93,358 customers (3.0%) are repeat buyers (Loyal 1,600 and Champions 1,201), so almost all revenue comes from one-time buyers. The largest segments are Needs attention (38.8%, last order about 269 days ago on average) and New (23.2%, about 90 days ago), while Lost customers are 19.5% of the base with an average of 474 days since the last order. Champions and Loyal customers spend the most per person (about 271 and 252 versus 136 for Needs attention), but the two groups are small and together bring only about 5.5% of revenue. New high value customers (15.5%, average spend 267) bring about 29% of revenue, which makes them the most valuable group to retain.

**Decision:** RFM counts delivered orders only. Frequency uses the 1 / 2 / 3+ rule because most people ordered once. Power BI reads `mart_customer_rfm` directly; no RFM logic in DAX. Retention is the main business question for this dataset: the "New" and "New high value" segments are the ones to convert into repeat buyers.